# Thermal-Contrast — train the U-Net, then run it on any `.mat`

```
TermoDataset video (T, H, W)  ->  extract_channels  ->  (4, H, W)  ->  U-Net  ->  mask
```

Data comes from every sub-dataset under `datasets/datasets_list` (Kaggle **and** TPU).
Checkpoints are pickles holding the model, optimizer, scheduler and epoch counter,
so `RESUME = "last"` continues a run exactly where it stopped.

In [ ]:
# === configuration ===
INCLUDE = None            # None -> every sub-dataset; or e.g. ["dataset_tpu"]
NUM_FRAMES = 64           # frames sampled for maxmin / maxfirst / std
EPOCHS = 30
BATCH_SIZE = 4
TEST_EVERY = 4            # every n-th video of each series goes to test
LR = 3e-4
WEIGHT_DECAY = 1e-4
POS_WEIGHT = 10.0         # defects are a small fraction of a frame
AUGMENT = True
NUM_WORKERS = 0
DEVICE = "auto"
RESUME = None             # None | "best" | "last" | path to a .pkl
PREVIEW_EVERY = 5

# section 4: any .mat video, not necessarily part of the dataset
MAT_PATH = "datasets/datasets_list/dataset_tpu/data/sample6.mat"
MAT_KEY = None            # None -> auto-detect the single 3-D array
THRESHOLD = 0.5

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models" / "Thermal-Contrast").is_dir())
sys.path.insert(0, str(ROOT / "models" / "Thermal-Contrast"))
import paths  # noqa: F401  puts repo root and models/ on sys.path

import matplotlib.pyplot as plt
import torch
from IPython.display import clear_output

from channels import CHANNEL_NAMES, CHANNEL_TITLES, ChannelParams
from common.device import get_device
from common.metrics import dice_score, iou_score
from data import build_datasets
from inference import evaluate_dataset, load_trained_model, mask_for_video, predict_channels, predict_mat
from train import CHECKPOINT_BEST, CHECKPOINT_LAST, train

PARAMS = ChannelParams(num_frames=NUM_FRAMES)
device = get_device(DEVICE)
print(f"device={device}  channels={list(CHANNEL_NAMES)}  params key={PARAMS.key}")

## 1. Split

The split is per video and per series (`R_`, `Z_`, `sample`), so no video appears on
both sides and both sub-datasets are represented in train and test.

In [ ]:
train_ds, test_ds = build_datasets(
    include=INCLUDE, params=PARAMS, test_every=TEST_EVERY, augment=AUGMENT
)
print(f"train {len(train_ds)} videos: {train_ds.video_ids}")
print(f"test  {len(test_ds)} videos: {test_ds.video_ids}")

channels, mask = train_ds[0]
print(f"\nitem: channels {tuple(channels.shape)} in [{channels.min():.2f}, {channels.max():.2f}], mask {tuple(mask.shape)}")

## 2. Train

The first run extracts channels for all videos and caches them under
`models/Thermal-Contrast/cache/`; later runs read the cache and start immediately.

In [ ]:
def plot_history(history, *, epoch, end_epoch, best_iou, best_epoch):
    if not history:
        return
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"epoch {epoch}/{end_epoch}   best test IoU {best_iou:.4f} @ {best_epoch}")

    axes[0].plot(epochs, [r["train_loss"] for r in history], label="train")
    axes[0].plot(epochs, [r["test_loss"] for r in history], label="test")
    axes[0].set_title("loss")

    axes[1].plot(epochs, [r["train_iou"] for r in history], label="train")
    axes[1].plot(epochs, [r["test_iou"] for r in history], label="test")
    axes[1].axhline(best_iou, color="tab:green", ls="--", lw=1)
    axes[1].set_title("IoU")

    axes[2].plot(epochs, [r["test_precision"] for r in history], label="precision")
    axes[2].plot(epochs, [r["test_recall"] for r in history], label="recall")
    axes[2].plot(epochs, [r["test_dice"] for r in history], label="dice")
    axes[2].set_title("test quality")

    for axis in axes:
        axis.set_xlabel("epoch")
        axis.grid(alpha=0.3)
        axis.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


def on_epoch_end(tracker, row, epoch, end_epoch, best_iou, best_epoch):
    if epoch % PREVIEW_EVERY and epoch != end_epoch:
        return
    clear_output(wait=True)
    plot_history(tracker.history, epoch=epoch, end_epoch=end_epoch, best_iou=best_iou, best_epoch=best_epoch)

In [ ]:
tracker, model, best_iou, best_epoch = train(
    epochs=EPOCHS,
    device=device,
    params=PARAMS,
    include=INCLUDE,
    test_every=TEST_EVERY,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    pos_weight=POS_WEIGHT,
    augment=AUGMENT,
    resume=RESUME,
    on_epoch_end=on_epoch_end,
)
print(f"best test IoU {best_iou:.4f} @ epoch {best_epoch}")
print(f"checkpoints: {CHECKPOINT_BEST}\n             {CHECKPOINT_LAST}")

## 3. Test set

Per-video scores, then the channel/GT/prediction grid for each test video.

In [ ]:
best = load_trained_model(CHECKPOINT_BEST, device)
rows = evaluate_dataset(best, test_ds, device, threshold=THRESHOLD)

print(f"{'video':12s} {'dice':>7s} {'iou':>7s} {'gt%':>7s} {'pred%':>7s}")
for row in rows:
    print(
        f"{row['video_id']:12s} {row['dice']:7.4f} {row['iou']:7.4f} "
        f"{100 * row['gt_positive']:7.2f} {100 * row['pred_positive']:7.2f}"
    )
print(f"\nmean dice={sum(r['dice'] for r in rows) / len(rows):.4f}  "
      f"iou={sum(r['iou'] for r in rows) / len(rows):.4f}")

In [ ]:
def show_prediction(channels, truth, prob, pred, title):
    fig, axes = plt.subplots(2, 3, figsize=(11, 7.4))
    for axis in axes.ravel():
        axis.axis("off")
    for column, name in enumerate(CHANNEL_NAMES[:3]):
        axes[0, column].imshow(channels[column].numpy(), cmap="inferno", vmin=0, vmax=1)
        axes[0, column].set_title(CHANNEL_TITLES[name])
    axes[1, 0].imshow(truth, cmap="gray", vmin=0, vmax=1)
    axes[1, 0].set_title("GT")
    axes[1, 1].imshow(channels[0].numpy(), cmap="inferno", vmin=0, vmax=1)
    axes[1, 1].contour(truth, levels=[0.5], colors="lime", linewidths=1.2)
    axes[1, 1].set_title("overlay")
    axes[1, 2].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[1, 2].set_title(f"pred Dice={dice_score(pred, truth):.3f}")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


for position, index in enumerate(test_ds.indices):
    channels, mask = test_ds.channels_and_mask(index)
    out = predict_channels(best, channels, device, threshold=THRESHOLD)
    truth = mask.squeeze().numpy()
    show_prediction(
        channels,
        truth,
        out["prob"],
        out["pred"],
        f"test[{position}] {test_ds.video_ids[position]}  IoU={iou_score(out['pred'], truth):.3f}",
    )

## 4. Any `.mat` video

`load_video_from_mat` reproduces the `TermoDataset` preprocessing exactly (read
`(H, W, T)` as float32, permute to `(T, H, W)`, bilinear resize to 256×256), then the
same `extract_channels` runs on it. Nothing about the file has to be known in
advance except that it holds one 3-D array.

In [ ]:
from video_io import mat_video_keys

mat_path = ROOT / MAT_PATH
print(f"{mat_path.name}: 3-D arrays = {mat_video_keys(mat_path)}")

result = predict_mat(best, mat_path, device, mat_key=MAT_KEY, params=PARAMS, threshold=THRESHOLD)
print(f"video {tuple(result['video'].shape)} -> channels {tuple(result['channels'].shape)}")
print(result["selection"].summary())

fig, axes = plt.subplots(1, 6, figsize=(19, 3.4))
for axis in axes:
    axis.axis("off")
for column, name in enumerate(CHANNEL_NAMES):
    axes[column].imshow(result["channels"][column].numpy(), cmap="inferno", vmin=0, vmax=1)
    axes[column].set_title(CHANNEL_TITLES[name])
axes[4].imshow(result["prob"], cmap="magma", vmin=0, vmax=1)
axes[4].set_title("probability")
axes[5].imshow(result["pred"], cmap="gray", vmin=0, vmax=1)
axes[5].set_title("prediction")
fig.suptitle(mat_path.name)
fig.tight_layout()
plt.show()

In [ ]:
# if the .mat happens to be one of the labelled videos, score the prediction
from inference import find_mask_for_mat

mask_path = find_mask_for_mat(mat_path)
if mask_path is None:
    print(f"no mask paired with {mat_path.name} \u2014 prediction only")
else:
    truth = mask_for_video(mask_path)
    print(f"{mask_path.name}: dice={dice_score(result['pred'], truth):.4f} "
          f"iou={iou_score(result['pred'], truth):.4f}")